In [6]:
import glob, os, re
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

BASE = "/Users/parin/Desktop/prosecutor explorer/plymouth"
RAW_DIR = f"{BASE}/raw"          # folder with the 3 DAMION xlsx files
OUT_DIR = f"{BASE}/raw_vs_clean"
os.makedirs(OUT_DIR, exist_ok=True)

RAW_YEARS = [2020, 2021, 2022]
MV_CHAPTERS = r"^(89|90|90A|90B|90C)/"   # motor-vehicle statutes

In [7]:
def norm_id(s):
    return s.astype("string").str.strip().str.upper().replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})

def norm_count(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def norm_date(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def pct(x):
    return (100 * x).round(1)

## Part 1: Raw data

In [8]:
RAW_RENAME = {
    "Docket Number": "docket", "Case ID Number": "case_id", "Race": "race", "Sex": "sex",
    "Offense Date": "offense_date", "Filing Date": "filing_date",
    "Court Location Description": "court", "Count": "count",
    "Filed Charge Code": "charge_code", "Filed Charge Description": "charge_desc",
    "Filed Charge Category": "severity", "Officer Agency Description": "agency",
    "Disp. Date": "disposition_date", "Disp. Code": "disposition_code",
    "Disposition Description": "disposition_desc", "Disp. Judge Name": "judge",
    "Case Status Description": "case_status",
}

frames = []
for f in sorted(glob.glob(os.path.join(RAW_DIR, "*Plymouth*.xlsx"))):
    d = pd.read_excel(f)
    d["file_year"] = int(re.search(r"(20\d\d)", f.split("Plymouth")[-1]).group(1))
    frames.append(d)
raw = pd.concat(frames, ignore_index=True).rename(columns=RAW_RENAME)

# drop the total row at the bottom of each file
footer = raw.docket.isna() & raw.charge_code.isna()
print("dropped footer rows:", footer.sum())
raw = raw[~footer].copy()

raw["docket"] = norm_id(raw.docket)
raw["case_id"] = norm_id(raw.case_id)
raw["count"] = norm_count(raw["count"])
for col in ["offense_date", "filing_date", "disposition_date"]:
    raw[col] = norm_date(raw[col])          # '00:00:00' junk -> NaT
raw["filing_year"] = raw.filing_date.dt.year.astype("Int64")
raw["is_mv"] = raw.charge_code.astype("string").str.match(MV_CHAPTERS, na=False)
raw["is_civil"] = raw.severity.eq("CM")
raw.shape

dropped footer rows: 3


(46429, 21)

In [9]:
raw.groupby("file_year").agg(rows=("case_id", "size"),
                             cases=("case_id", "nunique"),
                             dockets=("docket", "nunique"))

,rows,cases,dockets
file_year,,,
2020,14855,6176,6136
2021,15539,6406,6380
2022,16035,6678,6667


In [10]:
cols = [c for c in RAW_RENAME.values() if c in raw.columns]
(raw.groupby("file_year")[cols].apply(lambda x: x.isna().mean()) * 100).round(1).T

file_year,2020,2021,2022
docket,0.0,0.0,0.1
case_id,0.0,0.0,0.0
race,4.3,5.8,3.2
sex,0.3,0.5,0.4
offense_date,0.0,0.0,0.0
filing_date,0.0,0.0,0.0
court,0.0,0.0,0.0
count,0.0,0.0,0.0
charge_code,0.0,0.0,0.0
charge_desc,0.0,0.0,0.0


In [11]:
print("duplicate (case_id, count):", raw.duplicated(["case_id", "count"]).sum())
print("duplicate (docket, count): ", raw.duplicated(["docket", "count"]).sum())
print("cases in >1 file:          ", (raw.groupby("case_id").file_year.nunique() > 1).sum())

duplicate (case_id, count): 6
duplicate (docket, count):  270
cases in >1 file:           0


In [12]:
for col in ["race", "sex", "severity", "court", "case_status"]:
    display(pd.crosstab(raw[col].fillna("<NA>"), raw.file_year))

file_year,2020,2021,2022
race,,,
<NA>,646,897,508
A,42,57,98
B,4601,4274,4903
C,10,13,22
CNV,0,2,0
CV,25,28,17
H,175,189,346
I,15,6,28
O,11,11,14


file_year,2020,2021,2022
sex,,,
<NA>,45,72,60
F,3002,3427,3452
M,11806,12037,12513
U,2,3,10


file_year,2020,2021,2022
severity,,,
<NA>,432,351,369
CM,1534,1695,1883
F,4916,4813,4831
M+,6227,6745,7085
M-,1746,1935,1867


file_year,2020,2021,2022
court,,,
Brockton District Court,6833,6622,6956
Hingham District Court,1402,1454,1417
Plymouth District Court,3785,4110,3996
Wareham District Court,2835,3353,3666


file_year,2020,2021,2022
case_status,,,
Active,163,165,167
Closed,14274,14999,15336
Fugitive / Defaulted,379,310,442
Inactive / Warrant,37,54,85
Incompetent to Stand Trial,0,5,5
Not Guilty by Reason of Insanity,2,6,0


In [13]:
pd.crosstab(raw.disposition_desc.fillna("<NA>"), raw.file_year) \
  .sort_values(2020, ascending=False).head(20)

file_year,2020,2021,2022
disposition_desc,,,
Dismissed,7497,7538,7603
Continued w/o Finding,2035,2111,2489
Guilty Plea,1403,1520,1641
Nolle Prosequi,1100,1316,952
Not Responsible,1023,1148,1254
<NA>,621,540,682
Not Guilty Finding/Verdict,374,471,414
Pre-Trial Probation,266,160,334
Guilty Plea - Filed,118,158,133


In [14]:
raw.groupby("file_year")[["is_mv", "is_civil"]].agg(["sum", "mean"]).round(3)

is_mv        is_civil       
            sum   mean      sum   mean
file_year                             
2020       4740  0.319     1534  0.103
2021       5282   0.34     1695  0.109
2022       5873  0.366     1883  0.117

In [15]:
crim = raw[raw.groupby("case_id").is_civil.transform(lambda s: (~s).any())]
crim.groupby(["court", "filing_year"]).case_id.nunique().unstack(fill_value=0)

filing_year,2020,2021,2022,2023,2024,2025
court,,,,,,
Brockton District Court,2701,2775,2880,253,5,2
Hingham District Court,479,578,551,56,3,1
Plymouth District Court,1592,1745,1718,37,9,2
Wareham District Court,1041,1233,1555,38,2,1


## Part 2: Cleaned data

In [16]:
KEY = ["case_id", "charge_count_number"]
ch = pd.read_parquet(f"{BASE}/plymouth-charges-enriched.parquet").drop_duplicates()
ex = pd.read_parquet(f"{BASE}/plymouth-extras.parquet").drop_duplicates(subset=KEY)
cs = pd.read_parquet(f"{BASE}/plymouth-cases.parquet")

for name, d in [("charges", ch), ("extras", ex), ("cases", cs)]:
    print(f"{name}: {d.shape[0]:,} rows x {d.shape[1]} cols")
print("\ncharges columns:", list(ch.columns))

charges: 938,155 rows x 111 cols
extras: 937,944 rows x 10 cols
cases: 396,755 rows x 29 cols

charges columns: ['county', 'source_prr', 'case_id', 'defendant_id', 'docket_number', 'charge_count_number', 'num_charges_on_case', 'offense_date', 'filing_date', 'disposition_date', 'arrest_date', 'sentence_date', 'charge_code', 'charge_description', 'charge_category', 'court_name', 'filing_agency', 'icr_number', 'disposition_code', 'disposition_description', 'disposition_reason', 'disposition_event', 'disposition_comment', 'case_status', 'case_status_code', 'defendant_race', 'defendant_race_code', 'defendant_sex', 'defendant_sex_code', 'judge_name', 'rcmd_facility_code', 'rcmd_sentence_type', 'rcmd_sentence_date', 'rcmd_fine', 'rcmd_restitution', 'rcmd_jail_min', 'rcmd_jail_min_unit', 'rcmd_jail_max', 'rcmd_jail_max_unit', 'rcmd_susp_min', 'rcmd_susp_min_unit', 'rcmd_susp_max', 'rcmd_susp_max_unit', 'rcmd_actual_min', 'rcmd_actual_min_unit', 'rcmd_actual_max', 'rcmd_actual_max_unit', 'rcmd_

In [17]:
DISPO_MAP = {
    "DM": "dismissal", "NP": "dismissal", "NO": "dismissal", "NB": "dismissal", "DG": "dismissal", "D$": "dismissal",
    "DY": "declined", "DP": "declined", "NY": "declined",
    "DD": "diversion", "DC": "diversion", "PB": "diversion",
    "GP": "plea", "PF": "plea", "GL": "plea", "PL": "plea", "NC": "plea",
    "CW": "cwof",
    "GF": "trial_conviction", "GI": "trial_conviction", "VF": "trial_conviction", "FL": "trial_conviction",
    "NG": "acquittal", "DV": "acquittal", "NGI": "acquittal",
    "NR": "civil", "RF": "civil", "R$": "civil",
    "TF": "other", "FI": "other", "BO": "other", "RM": "other", "AW": "other",
    "AF": "other", "IN": "other", "NI": "other", "CC": "other",
}
ch["outcome"] = ch.disposition_code.map(DISPO_MAP)
ch.loc[ch.disposition_code.isna(), "outcome"] = "pending"
ch["outcome"] = ch.outcome.fillna("other")

In [18]:
ch["filing_date"] = norm_date(ch.filing_date)
cl = ch[ch.filing_date.dt.year.isin(RAW_YEARS)].copy()
cl["key"] = norm_id(cl.case_id)
cl["count"] = norm_count(cl.charge_count_number)
cl["filing_year"] = cl.filing_date.dt.year

print(f"clean charges in window: {len(cl):,}")
if "source" in cl:
    print("source:", cl.source.value_counts().to_dict())
print("duplicate (case_id, count):", cl.duplicated(["key", "count"]).sum())

clean charges in window: 46,227
duplicate (case_id, count): 4


## Part 3: Overlap

In [19]:
clean_ids = set(norm_id(ch.case_id).dropna())
n_cid = len(clean_ids & set(raw.case_id.dropna()))
n_dkt = len(clean_ids & set(raw.docket.dropna()))
print("clean case_id sample:", list(clean_ids)[:5])
print(f"match raw Case ID: {n_cid:,} | match raw Docket: {n_dkt:,}")

RAW_KEY = "docket" if n_dkt > n_cid else "case_id"
print("joining on raw", RAW_KEY)

r = raw.assign(key=raw[RAW_KEY])
r_w = r[r.filing_year.isin(RAW_YEARS)]

clean case_id sample: ['22-005358', '02-002738', 'BDA-947785-01', 'BDA-953525-01', 'PDJ-971737-01']
match raw Case ID: 19,260 | match raw Docket: 0
joining on raw case_id


In [20]:
by_year = pd.DataFrame({
    "raw": r_w.groupby("filing_year").key.nunique(),
    "clean": cl.groupby("filing_year").key.nunique(),
})
by_year["diff"] = by_year.raw - by_year.clean
display(by_year)

rk, ck = set(r_w.key.dropna()), set(cl.key.dropna())
print(f"in both: {len(rk & ck):,} | raw only: {len(rk - ck):,} | clean only: {len(ck - rk):,}")
print(f"raw-only cases found in clean under another filing year: {len((rk - ck) & clean_ids):,}")

,raw,clean,diff
filing_year,,,
2020,5813,6070,-257
2021,6332,6353,-21
2022,6706,6713,-7


in both: 18,851 | raw only: 0 | clean only: 285
raw-only cases found in clean under another filing year: 0


In [21]:
m = r_w.merge(cl, on=["key", "count"], how="outer", suffixes=("_raw", "_clean"), indicator=True)
display(m._merge.value_counts())
both = m[m._merge.eq("both")].copy()

_merge
both          45548
right_only      689
left_only         0
Name: count, dtype: int64

In [22]:
def compare(a, b, date=False):
    if date:
        a, b = norm_date(a), norm_date(b)
    else:
        a, b = a.astype("string").str.strip().str.upper(), b.astype("string").str.strip().str.upper()
    return (a == b) | (a.isna() & b.isna()), a, b

rows = []
for f, is_date in [("filing_date", True), ("disposition_date", True),
                   ("disposition_code", False), ("charge_code", False)]:
    if f"{f}_raw" not in both or f"{f}_clean" not in both:
        continue
    ok, a, b = compare(both[f"{f}_raw"], both[f"{f}_clean"], is_date)
    rows.append({"field": f, "agree_%": pct(ok.mean()),
                 "raw_blank_clean_filled": int((a.isna() & b.notna()).sum()),
                 "raw_filled_clean_blank": int((a.notna() & b.isna()).sum()),
                 "disagree": int((~ok).sum())})
    if (~ok).any():
        both.loc[~ok, ["key", "count", f"{f}_raw", f"{f}_clean"]] \
            .to_csv(f"{OUT_DIR}/mismatch_{f}.csv", index=False)
pd.DataFrame(rows)

,field,agree_%,raw_blank_clean_filled,raw_filled_clean_blank,disagree
0,filing_date,100.0,0,0,0
1,disposition_date,100.0,0,0,0
2,disposition_code,100.0,0,0,8
3,charge_code,100.0,0,0,10


In [23]:
pairs = [("race_raw", "race_clean"), ("sex_raw", "sex_clean"),
         ("severity", "sev_class"), ("is_civil_raw", "is_civil_clean"), ("court", "court_name")]
for a, b in pairs:
    if a in both and b in both:
        print(f"\n{a} -> {b}")
        display(pd.crosstab(both[a].fillna("<NA>"), both[b].fillna("<NA>")))
    else:
        print(f"\nskipped {a} -> {b} (column not in cleaned file)")


skipped race_raw -> race_clean (column not in cleaned file)

skipped sex_raw -> sex_clean (column not in cleaned file)

skipped severity -> sev_class (column not in cleaned file)

skipped is_civil_raw -> is_civil_clean (column not in cleaned file)

court -> court_name


court_name,Brockton District Court,Hingham District Court,Plymouth District Court,Wareham District Court
court,,,,
Brockton District Court,19876,0,0,0
Hingham District Court,0,4125,0,0
Plymouth District Court,0,0,11787,0
Wareham District Court,0,0,0,9760


In [24]:
t = pd.crosstab(both.disposition_desc.fillna("<NA>"), both.outcome.fillna("<NA>"))
t.loc[t.sum(axis=1).sort_values(ascending=False).index]

outcome,acquittal,civil,cwof,declined,dismissal,diversion,other,pending,plea,trial_conviction
disposition_desc,,,,,,,,,,
Dismissed,0,1,1,0,22213,0,0,0,1,0
Continued w/o Finding,0,1,6521,0,1,0,0,0,0,0
Guilty Plea,0,0,0,0,1,0,0,0,4501,0
Nolle Prosequi,0,0,0,0,3342,0,0,0,0,0
Not Responsible,0,3284,1,0,1,0,0,0,0,0
<NA>,0,0,0,0,0,0,0,1798,0,0
Not Guilty Finding/Verdict,1236,0,0,0,0,0,0,0,0,0
Pre-Trial Probation,0,0,0,0,0,745,0,0,0,0
Guilty Finding/Verdict,0,0,0,0,0,0,0,0,0,439


In [26]:
e = ex.assign(key=norm_id(ex.case_id), count=norm_count(ex.charge_count_number))[["key", "count"]]
hit = r_w.merge(e, on=["key", "count"], how="left", indicator=True)._merge.eq("both")
print(f"raw charges with an extras row: {hit.sum():,} / {len(r_w):,} ({pct(hit.mean())}%)")

case_ids = set(norm_id(cs.case_id).dropna()) if "case_id" in cs else set()
print(f"raw cases in cases table:       {len(rk & case_ids):,} / {len(rk):,}")

raw charges with an extras row: 45,540 / 45,540 (100.0%)
raw cases in cases table:       18,851 / 18,851


In [27]:
m[m._merge.eq("left_only")].to_csv(f"{OUT_DIR}/raw_only_charges.csv", index=False)
m[m._merge.eq("right_only")].to_csv(f"{OUT_DIR}/clean_only_charges.csv", index=False)
print("saved to", OUT_DIR)

saved to /Users/parin/Desktop/prosecutor explorer/plymouth/raw_vs_clean
